# Tests: Verifying if torch is well installed and that the data is usable

In [2]:
import torch
print(torch.__version__, torch.cuda.is_available())

# a model with ONE weight: prediction = w * x
w = torch.tensor(2.0, requires_grad=True)   # requires_grad: "track this for gradients"
x, y = torch.tensor(3.0), torch.tensor(12.0)  # input 3, true answer 12

loss = (w * x - y) ** 2     # squared error
loss.backward()             # compute the gradient automatically
print(loss.item(), w.grad.item())

2.15.0+cu132 True
36.0 -36.0


In [3]:
w = torch.tensor(2.0, requires_grad=True)
x, y = torch.tensor(3.0), torch.tensor(12.0)
lr = 0.2

for step in range(30):
    loss = (w * x - y) ** 2
    loss.backward()                 # compute the gradient
    with torch.no_grad():           # update w without tracking the update itself
        w -= lr * w.grad            # nudge w against the gradient
    w.grad.zero_()                  # reset: gradients would otherwise pile up
    if step % 5 == 0:
        print(step, round(w.item(), 3), round(loss.item(), 3))

0 9.2 36.0
5 -613.832 508201.719
10 73410.906 7174136320.0
15 -8721747.0 101275219787776.0
20 1036263936.0 1.4296724296087634e+18
25 -123122401280.0 2.018226698871174e+22


In [4]:
import torch_geometric, torch
print(torch_geometric.__version__, torch.__version__)   

C:\Users\Rayene\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Rayene\AppData\Roaming\Python\Python314\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


2.8.1 2.15.0+cu132


In [5]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only")

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [6]:
%pip show torch_geometric

Name: torch-geometric
Version: 2.8.1
Summary: Graph Neural Network Library for PyTorch
Home-page: https://pyg.org
Author: 
Author-email: Matthias Fey <matthias@pyg.org>
License-Expression: MIT
Location: C:\Users\Rayene\AppData\Roaming\Python\Python314\site-packages
Requires: aiohttp, fsspec, jinja2, numpy, psutil, pyparsing, requests, tqdm, xxhash
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch_geometric
print(torch_geometric.__version__)

2.8.1


In [8]:
import sys; print(sys.executable)

c:\python314\python.exe


In [9]:
import torch
print(torch.__version__, torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only")

2.15.0+cu132 True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [10]:
import torch, torch_geometric
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch_geometric.__version__, torch.__version__, device)

2.8.1 2.15.0+cu132 cuda


In [11]:
import zipfile, os, shutil

def peek(path, top=25):
    with zipfile.ZipFile(path) as z:
        infos = z.infolist()
        total = sum(i.file_size for i in infos)
        print(os.path.basename(path), len(infos), "files,", round(total / 1e9, 2), "GB uncompressed")
        for i in sorted(infos, key=lambda i: i.file_size, reverse=True)[:top]:
            print(round(i.file_size / 1e6, 1), "MB", i.filename)

peek(r"data/archive.zip")      # the 12 GB one
peek(r"data/BABD.zip")           # the 5.3 GB one
print(round(shutil.disk_usage("C:\\").free / 1e9, 1), "GB free on C:")

archive.zip 112804 files, 77.57 GB uncompressed
594.4 MB ETHdata/ETHdata/0x6131b5fae19ea4f9d964eac0408e4408b66337b5/normal_transactions.json
587.5 MB Sup-CATS-Txs/Sup-CATS-Txs/0x5132a183e9f3cb7c848b0aac5ae0c4f0491b7ab2/normal_transactions.json
577.3 MB Sup-CATS-Txs/Sup-CATS-Txs/0xaeba8e2307a22b6824a9a7a39f8b016c357cd1fe/normal_transactions.json
562.3 MB Sup-CATS-Txs/Sup-CATS-Txs/0xa76e31d8471d569efdd3d95d1b11ce6710f4533f/normal_transactions.json
430.8 MB Sup-CATS-Txs/Sup-CATS-Txs/0x01c3a1a6890a146ac187a019f9863b3ab2bff91e/normal_transactions.json
394.2 MB Sup-CATS-Txs/Sup-CATS-Txs/0x38cdb1a8207264c1a07c42c43a4c3ed4bfab7cea/normal_transactions.json
317.2 MB Sup-CATS-Txs/Sup-CATS-Txs/0xff00000000000000000000000000000000000255/normal_transactions.json
317.2 MB Sup-CATS-Txs/Sup-CATS-Txs/0x41b8cd6791de4d8f9e0eaf7861ac506822adce12/normal_transactions.json
311.5 MB Sup-CATS-Txs/Sup-CATS-Txs/0x39da41747a83aee658334415666f3ef92dd0d541/normal_transactions.json
273.1 MB ETHdata/ETHdata/0x3fc91a3a

In [13]:
import zipfile
from collections import defaultdict

ARCHIVE = r"data/archive.zip"      # <- the Real-CATS zip
with zipfile.ZipFile(ARCHIVE) as z:
    agg = defaultdict(lambda: [0, 0])
    shallow = []
    for i in z.infolist():
        if i.is_dir():
            continue
        top = i.filename.split("/")[0]
        agg[top][0] += 1
        agg[top][1] += i.file_size
        if i.filename.count("/") <= 1:
            shallow.append((round(i.file_size / 1e6, 1), i.filename))

for k, (n, s) in sorted(agg.items(), key=lambda kv: -kv[1][1]):
    print(k, n, "files", round(s / 1e9, 2), "GB")
print()
for size, name in sorted(shallow, key=lambda t: t[1])[:60]:
    print(size, "MB", name)

Sup-CATS-Txs 42630 files 54.09 GB
ETHdata 70174 files 23.48 GB



In [15]:
import os
BABD = r"data/BABD.zip"            # <- the BABD zip
os.makedirs(r"data/babd", exist_ok=True)
with zipfile.ZipFile(BABD) as z:
    z.extract("BABD-13.csv", r"data/babd")
print(os.path.getsize(r"data/babd/BABD-13.csv") / 1e6, "MB")

785.101222 MB


# Step 1 : MLP on the elliptic

In [16]:
import pandas as pd, numpy as np, torch
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA = "../elliptic"        # <- folder with the Elliptic CSV files

feats = pd.read_csv(f"{DATA}/elliptic_txs_features.csv", header=None)
classes = pd.read_csv(f"{DATA}/elliptic_txs_classes.csv")
feats.columns = ["txId", "time_step"] + [f"f{i}" for i in range(165)]

d = feats.merge(classes, on="txId")
d = d[d["class"] != "unknown"].copy()
d["y"] = (d["class"] == "1").astype(int)       # 1 = illicit, 0 = licit

tr = d[d["time_step"] <= 34]                   # temporal split
te = d[d["time_step"] > 34]
xcols = [c for c in d.columns if c.startswith("f")]

scaler = StandardScaler().fit(tr[xcols])       # fit on the training part only
Xtr_t = torch.tensor(scaler.transform(tr[xcols]), dtype=torch.float32, device=device)
Xte_t = torch.tensor(scaler.transform(te[xcols]), dtype=torch.float32, device=device)
ytr_t = torch.tensor(tr["y"].values, dtype=torch.float32, device=device)
yte_t = torch.tensor(te["y"].values, dtype=torch.float32, device=device)

print(Xtr_t.shape, Xte_t.shape, Xtr_t.device)
print(round(ytr_t.mean().item(), 4), round(yte_t.mean().item(), 4)) 

torch.Size([29894, 165]) torch.Size([16670, 165]) cuda:0
0.1158 0.065


Running both MLP and Forest to compare

In [17]:
import torch.nn as nn
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(165, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
).to(device)

pos_weight = ((ytr_t == 0).sum() / (ytr_t == 1).sum()).reshape(1)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(300):
    model.train()
    opt.zero_grad()
    loss = loss_fn(model(Xtr_t).squeeze(1), ytr_t)
    loss.backward()
    opt.step()
    if epoch % 50 == 0 or epoch == 299:
        print(epoch, round(loss.item(), 4))

model.eval()
with torch.no_grad():
    proba = torch.sigmoid(model(Xte_t).squeeze(1)).cpu().numpy()
y_te = yte_t.cpu().numpy().astype(int)
pred = (proba >= 0.5).astype(int)
print("===== MLP")
print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, digits=3))
print("PR AUC", round(average_precision_score(y_te, proba), 4))

# Random Forest on exactly the same split and features, as the reference
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(Xtr_t.cpu().numpy(), tr["y"].values)
rf_proba = rf.predict_proba(Xte_t.cpu().numpy())[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)
print("===== Random Forest")
print(confusion_matrix(y_te, rf_pred))
print(classification_report(y_te, rf_pred, digits=3))
print("PR AUC", round(average_precision_score(y_te, rf_proba), 4))

0 1.2297
50 0.4786
100 0.2853
150 0.2102
200 0.1633
250 0.1268
299 0.099
===== MLP
[[15198   389]
 [  503   580]]
              precision    recall  f1-score   support

           0      0.968     0.975     0.971     15587
           1      0.599     0.536     0.565      1083

    accuracy                          0.946     16670
   macro avg      0.783     0.755     0.768     16670
weighted avg      0.944     0.946     0.945     16670

PR AUC 0.5277
===== Random Forest
[[15557    30]
 [  300   783]]
              precision    recall  f1-score   support

           0      0.981     0.998     0.990     15587
           1      0.963     0.723     0.826      1083

    accuracy                          0.980     16670
   macro avg      0.972     0.861     0.908     16670
weighted avg      0.980     0.980     0.979     16670

PR AUC 0.7934


We can clearly see that random forest is way more effective than MLP

In [18]:
import copy
from sklearn.metrics import precision_recall_curve

fit_mask = torch.tensor((tr["time_step"] <= 29).values, device=device)
Xf, yf = Xtr_t[fit_mask], ytr_t[fit_mask]          # time steps 1-29: training
Xv, yv = Xtr_t[~fit_mask], ytr_t[~fit_mask]        # time steps 30-34: validation
yv_np = yv.cpu().numpy().astype(int)

torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(165, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(32, 1),
).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=((yf == 0).sum() / (yf == 1).sum()).reshape(1))
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_ap, best_state, best_epoch = -1, None, -1
for epoch in range(500):
    model.train(); opt.zero_grad()
    loss = loss_fn(model(Xf).squeeze(1), yf); loss.backward(); opt.step()
    if epoch % 10 == 0:
        model.eval()                                # turns dropout off for evaluation
        with torch.no_grad():
            pv = torch.sigmoid(model(Xv).squeeze(1)).cpu().numpy()
        ap = average_precision_score(yv_np, pv)
        if ap > best_ap:
            best_ap, best_state, best_epoch = ap, copy.deepcopy(model.state_dict()), epoch
        if epoch % 50 == 0:
            print(epoch, round(loss.item(), 4), "val PR AUC", round(ap, 4))
print("best epoch", best_epoch, "val PR AUC", round(best_ap, 4))

model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    pv = torch.sigmoid(model(Xv).squeeze(1)).cpu().numpy()
    proba2 = torch.sigmoid(model(Xte_t).squeeze(1)).cpu().numpy()
p, r, t = precision_recall_curve(yv_np, pv)
thr = t[np.argmax((2 * p * r / (p + r + 1e-12))[:-1])]
pred2 = (proba2 >= thr).astype(int)

print("===== MLP with validation, early stopping, dropout; threshold", round(thr, 3))
print(confusion_matrix(y_te, pred2))
print(classification_report(y_te, pred2, digits=3))
print("PR AUC", round(average_precision_score(y_te, proba2), 4))

0 1.2416 val PR AUC 0.2046
50 0.4895 val PR AUC 0.699
100 0.305 val PR AUC 0.7868
150 0.2344 val PR AUC 0.774
200 0.2023 val PR AUC 0.7836
250 0.1677 val PR AUC 0.7976
300 0.15 val PR AUC 0.8158
350 0.1348 val PR AUC 0.8345
400 0.1219 val PR AUC 0.8508
450 0.1069 val PR AUC 0.8602
best epoch 490 val PR AUC 0.8643
===== MLP with validation, early stopping, dropout; threshold 0.534
[[14823   764]
 [  527   556]]
              precision    recall  f1-score   support

           0      0.966     0.951     0.958     15587
           1      0.421     0.513     0.463      1083

    accuracy                          0.923     16670
   macro avg      0.693     0.732     0.711     16670
weighted avg      0.930     0.923     0.926     16670

PR AUC 0.2914


We can see that the second mlp version is worse than the first.
For the result we will use the first version and summerize the tests in the next cell 

In [19]:
from sklearn.metrics import recall_score, precision_score, f1_score

te_steps = te["time_step"].values
rows = []
for s in sorted(np.unique(te_steps)):
    m = te_steps == s
    rows.append({"step": s, "n_illicit": int(y_te[m].sum()),
                 "rf_recall": recall_score(y_te[m], rf_pred[m], zero_division=0),
                 "mlp_recall": recall_score(y_te[m], pred[m], zero_division=0)})
print(pd.DataFrame(rows).round(2).to_string(index=False))

def summarize(dataset, name, y, yhat, score):
    return {"dataset": dataset, "model": name,
            "precision": round(precision_score(y, yhat), 3),
            "recall": round(recall_score(y, yhat), 3),
            "f1": round(f1_score(y, yhat), 3),
            "pr_auc": round(average_precision_score(y, score), 3)}

results_dl = pd.DataFrame([
    summarize("Elliptic", "Random Forest", y_te, rf_pred, rf_proba),
    summarize("Elliptic", "MLP", y_te, pred, proba),
    summarize("Elliptic", "MLP + val/early stop/dropout", y_te, pred2, proba2),
])
print(results_dl.to_string(index=False))

 step  n_illicit  rf_recall  mlp_recall
   35        182       0.97        0.92
   36         33       1.00        0.67
   37         40       0.65        0.60
   38        111       0.90        0.64
   39         81       0.93        0.47
   40        112       0.64        0.51
   41        116       0.93        0.39
   42        239       0.79        0.64
   43         24       0.00        0.00
   44         24       0.04        0.04
   45          5       0.00        0.00
   46          2       0.50        0.00
   47         22       0.00        0.00
   48         36       0.00        0.00
   49         56       0.02        0.00
 dataset                        model  precision  recall    f1  pr_auc
Elliptic                Random Forest      0.963   0.723 0.826   0.793
Elliptic                          MLP      0.599   0.536 0.565   0.528
Elliptic MLP + val/early stop/dropout      0.421   0.513 0.463   0.291


# Step 2 GNN on Elliptic

In [20]:
from torch_geometric.data import Data

edges = pd.read_csv(f"{DATA}/elliptic_txs_edgelist.csv")
all_ = feats.merge(classes, on="txId", how="left")          # all 203,769 nodes, labelled or not
idx = pd.Series(np.arange(len(all_)), index=all_["txId"])
src = idx[edges["txId1"]].values
dst = idx[edges["txId2"]].values
edge_index = torch.tensor(np.vstack([np.r_[src, dst], np.r_[dst, src]]), dtype=torch.long)   # both directions

lab = all_["class"].map({"1": 1, "2": 0}).fillna(-1).astype(int).values   # -1 = unknown
step = all_["time_step"].values

scaler_g = StandardScaler().fit(all_.loc[step <= 34, xcols])               # training period only
x = torch.tensor(scaler_g.transform(all_[xcols]), dtype=torch.float32)

train_mask = torch.tensor((lab >= 0) & (step <= 34))
test_mask  = torch.tensor((lab >= 0) & (step > 34))
data = Data(x=x, edge_index=edge_index, y=torch.tensor(lab)).to(device)
train_mask, test_mask = train_mask.to(device), test_mask.to(device)

print(data)
print(int(train_mask.sum()), int(test_mask.sum()))
print("edges crossing time steps:", int((step[src] != step[dst]).sum()))

Data(x=[203769, 165], edge_index=[2, 468710], y=[203769])
29894 16670
edges crossing time steps: 0


In [21]:
from torch_geometric.nn import GCNConv, SAGEConv
import torch.nn.functional as F

class GNN(nn.Module):
    def __init__(self, conv, in_dim=165, hid=64, p=0.2):
        super().__init__()
        self.c1, self.c2 = conv(in_dim, hid), conv(hid, hid)
        self.out = nn.Linear(hid, 1)
        self.p = p
    def forward(self, x, ei):
        h = F.dropout(F.relu(self.c1(x, ei)), self.p, self.training)
        h = F.dropout(F.relu(self.c2(h, ei)), self.p, self.training)
        return self.out(h).squeeze(1)

y_f = data.y.float()
pw = ((data.y[train_mask] == 0).sum() / (data.y[train_mask] == 1).sum()).float().reshape(1)

def run(conv, name, epochs=300, lr=5e-3):
    torch.manual_seed(42)
    m = GNN(conv).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=5e-4)
    lf = nn.BCEWithLogitsLoss(pos_weight=pw)
    for ep in range(epochs):
        m.train(); opt.zero_grad()
        out = m(data.x, data.edge_index)
        loss = lf(out[train_mask], y_f[train_mask])     # loss on training nodes only
        loss.backward(); opt.step()
        if ep % 50 == 0 or ep == epochs - 1:
            print(name, ep, round(loss.item(), 4))
    m.eval()
    with torch.no_grad():
        sc = torch.sigmoid(m(data.x, data.edge_index))[test_mask].cpu().numpy()
    yt = data.y[test_mask].cpu().numpy()
    yh = (sc >= 0.5).astype(int)
    print("=====", name)
    print(confusion_matrix(yt, yh))
    print(classification_report(yt, yh, digits=3))
    print("PR AUC", round(average_precision_score(yt, sc), 4))
    return m, sc, yh, yt

gcn,  gcn_sc,  gcn_pred,  yt = run(GCNConv,  "GCN")
sage, sage_sc, sage_pred, _  = run(SAGEConv, "GraphSAGE")

print("same test order as step 1:", (yt == y_te).all())
results_dl = pd.concat([results_dl, pd.DataFrame([
    summarize("Elliptic", "GCN", yt, gcn_pred, gcn_sc),
    summarize("Elliptic", "GraphSAGE", yt, sage_pred, sage_sc),
])], ignore_index=True)
print(results_dl.to_string(index=False))

GCN 0 1.2613
GCN 50 0.3136
GCN 100 0.194
GCN 150 0.1329
GCN 200 0.1081
GCN 250 0.0921
GCN 299 0.0861
===== GCN
[[15433   154]
 [  511   572]]
              precision    recall  f1-score   support

           0      0.968     0.990     0.979     15587
           1      0.788     0.528     0.632      1083

    accuracy                          0.960     16670
   macro avg      0.878     0.759     0.806     16670
weighted avg      0.956     0.960     0.956     16670

PR AUC 0.6488
GraphSAGE 0 1.2149
GraphSAGE 50 0.1434
GraphSAGE 100 0.0575
GraphSAGE 150 0.0361
GraphSAGE 200 0.0255
GraphSAGE 250 0.019
GraphSAGE 299 0.0177
===== GraphSAGE
[[15435   152]
 [  450   633]]
              precision    recall  f1-score   support

           0      0.972     0.990     0.981     15587
           1      0.806     0.584     0.678      1083

    accuracy                          0.964     16670
   macro avg      0.889     0.787     0.829     16670
weighted avg      0.961     0.964     0.961     16670



In [22]:
pre, post = te_steps < 43, te_steps >= 43
models_pred = {"Random Forest": rf_pred, "MLP": pred, "GCN": gcn_pred, "GraphSAGE": sage_pred}
for name, p in models_pred.items():
    print(f"{name:14s} recall steps 35-42: {recall_score(y_te[pre], p[pre]):.3f}"
          f"   steps 43-49: {recall_score(y_te[post], p[post]):.3f}")

Random Forest  recall steps 35-42: 0.853   steps 43-49: 0.018
MLP            recall steps 35-42: 0.633   steps 43-49: 0.006
GCN            recall steps 35-42: 0.625   steps 43-49: 0.006
GraphSAGE      recall steps 35-42: 0.690   steps 43-49: 0.012


# Step 3: addresses on BABD-13

In [23]:
import pandas as pd

babd = pd.read_csv(r"C:\data\babd\BABD-13.csv")
print(babd.shape)
cols = list(babd.columns)
print(cols[:12], "...", cols[-8:])
print("non-numeric:", babd.select_dtypes(exclude="number").columns.tolist())

few = [c for c in cols if babd[c].nunique() <= 20]
print("few distinct values:", few)
for c in few[:4]:
    print(babd[c].value_counts().sort_index())

print("time-like columns:", [c for c in cols if any(k in c.lower() for k in ["time", "block", "height", "date"])])
print("missing values:", int(babd.isna().sum().sum()))

(544462, 151)
['account', 'SW', 'PAIa11-1', 'PAIa11-2', 'PAIa12', 'PAIa13', 'PAIa14-1', 'PAIa14-2', 'PAIa14-3', 'PAIa14-4', 'PAIa14-R1', 'PAIa14-R2'] ... ['S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'label']
non-numeric: ['account', 'SW']
few distinct values: ['SW', 'S6', 'label']
SW
SA    532732
WA     11730
Name: count, dtype: int64
S6
1        15
2       371
3     40656
4     21730
5    364620
6     19851
7     80421
8     16798
Name: count, dtype: int64
label
0       8686
1      91617
2      13861
3     300000
4        180
5       9309
6     105257
7         27
8         16
9         15
10      1580
11     12412
12      1502
Name: count, dtype: int64
time-like columns: []
missing values: 0


In [24]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 1.0/48.9 MB 8.2 MB/s eta 0:00:06
   -- ------------------------------------- 2.6/48.9 MB 8.0 MB/s eta 0:00:06
   --- ------------------------------------ 3.9/48.9 MB 7.5 MB/s eta 0:00:07
   ---- ----------------------------------- 5.8/48.9 MB 7.6 MB/s eta 0:00:06
   ----- ---------------------------------- 7.3/48.9 MB 7.6 MB/s eta 0:00:06
   ------- -------------------------------- 8.9/48.9 MB 7.6 MB/s eta 0:00:06
   -------- ------------------------------- 10.5/48.9 MB 7.6 MB/s eta 0:00:06
   --------- ------------------------------ 11.8/48.9 MB 7.4 MB/s eta 0:00:06
   ---------- ----------------------------- 13.1/48.9 MB 7.2 MB/s eta 0:00:05
   ----------- ---------------------------- 14.4/48.9 MB 7.1 MB/s eta 0:00:05
   ------------ --------------------------- 15.7/48.9 MB 7.1 MB/s eta 0:00:05



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier

names = ["Blackmail", "Cyber-security service", "Darknet market", "Centralized exchange",
         "P2P financial infrastructure", "P2P financial service", "Gambling",
         "Government criminal blacklist", "Money laundering", "Ponzi scheme",
         "Mining pool", "Tumbler", "Individual wallet"]

feat_cols = [c for c in babd.columns if c not in ("account", "SW", "label")]
Xb = babd[feat_cols].values.astype("float32")
yb = babd["label"].values
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, stratify=yb, random_state=42)
print(len(feat_cols), Xb_tr.shape, Xb_te.shape)
print("test count per class:", np.bincount(yb_te))

xgb = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                    tree_method="hist", device="cuda", random_state=42)
xgb.fit(Xb_tr, yb_tr)
xp = xgb.predict(Xb_te)
print("accuracy", round(accuracy_score(yb_te, xp), 4),
      "macro F1", round(f1_score(yb_te, xp, average="macro"), 4),
      "weighted F1", round(f1_score(yb_te, xp, average="weighted"), 4))
print(classification_report(yb_te, xp, target_names=names, digits=3, zero_division=0))

148 (435569, 148) (108893, 148)
test count per class: [ 1737 18324  2772 60000    36  1862 21052     5     3     3   316  2483
   300]


C:\Users\Rayene\AppData\Roaming\Python\Python314\site-packages\xgboost\core.py:569: UserWarning: [15:27:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


accuracy 0.9698 macro F1 0.6617 weighted F1 0.9686
                               precision    recall  f1-score   support

                    Blackmail      0.615     0.559     0.586      1737
       Cyber-security service      0.944     0.961     0.952     18324
               Darknet market      0.990     0.994     0.992      2772
         Centralized exchange      0.987     0.992     0.989     60000
 P2P financial infrastructure      1.000     0.917     0.957        36
        P2P financial service      0.990     0.941     0.965      1862
                     Gambling      0.981     0.978     0.979     21052
Government criminal blacklist      0.000     0.000     0.000         5
             Money laundering      0.250     0.333     0.286         3
                 Ponzi scheme      0.000     0.000     0.000         3
                  Mining pool      0.987     0.953     0.969       316
                      Tumbler      0.885     0.833     0.858      2483
            Individual wa

In [26]:
import copy
from torch.utils.data import TensorDataset, DataLoader

def slog(a):
    return np.sign(a) * np.log1p(np.abs(a))

Xb_fit, Xb_val, yb_fit, yb_val = train_test_split(Xb_tr, yb_tr, test_size=0.1, stratify=yb_tr, random_state=42)
sc_b = StandardScaler().fit(slog(Xb_fit))
to_t = lambda a: torch.tensor(sc_b.transform(slog(a)), dtype=torch.float32)
Xf_b, Xv_b, Xt_b = to_t(Xb_fit), to_t(Xb_val), to_t(Xb_te)
loader = DataLoader(TensorDataset(Xf_b, torch.tensor(yb_fit)), batch_size=1024, shuffle=True)

counts = np.bincount(yb_fit, minlength=13)
w = (counts.sum() / (13 * counts)) ** 0.5
w = torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

torch.manual_seed(42)
mlp_b = nn.Sequential(
    nn.Linear(148, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 13),
).to(device)
opt = torch.optim.Adam(mlp_b.parameters(), lr=1e-3)
lf = nn.CrossEntropyLoss(weight=w)

def predict(m, X):
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), 8192):
            out.append(m(X[i:i + 8192].to(device)).argmax(1).cpu())
    return torch.cat(out).numpy()

best, best_state, best_ep = -1, None, -1
for epoch in range(30):
    mlp_b.train()
    for xb_, yb_ in loader:
        xb_, yb_ = xb_.to(device), yb_.to(device)
        opt.zero_grad(); loss = lf(mlp_b(xb_), yb_); loss.backward(); opt.step()
    vf = f1_score(yb_val, predict(mlp_b, Xv_b), average="macro")
    if vf > best:
        best, best_state, best_ep = vf, copy.deepcopy(mlp_b.state_dict()), epoch
    print(epoch, round(loss.item(), 4), "val macro F1", round(vf, 4))

mlp_b.load_state_dict(best_state)
mp = predict(mlp_b, Xt_b)
print("best epoch", best_ep)
print(classification_report(yb_te, mp, target_names=names, digits=3, zero_division=0))

big = [c for c in range(13) if c not in (7, 8, 9)]          # without the three tiny classes
def summ_mc(name, y, p):
    return {"dataset": "BABD-13", "model": name,
            "accuracy": round(accuracy_score(y, p), 4),
            "macro_f1": round(f1_score(y, p, average="macro"), 4),
            "macro_f1_10cls": round(f1_score(y, p, labels=big, average="macro"), 4),
            "weighted_f1": round(f1_score(y, p, average="weighted"), 4)}
results_babd = pd.DataFrame([summ_mc("XGBoost", yb_te, xp), summ_mc("MLP", yb_te, mp)])
print(results_babd.to_string(index=False))

0 0.7268 val macro F1 0.4317
1 0.7258 val macro F1 0.4809
2 0.5599 val macro F1 0.5077
3 0.5732 val macro F1 0.5042
4 0.5219 val macro F1 0.5342
5 0.4541 val macro F1 0.6235
6 0.4873 val macro F1 0.5201
7 0.4993 val macro F1 0.5278
8 0.4582 val macro F1 0.5767
9 0.4343 val macro F1 0.6197
10 0.3579 val macro F1 0.5705
11 0.4865 val macro F1 0.6483
12 0.5998 val macro F1 0.6486
13 0.5087 val macro F1 0.5829
14 0.4831 val macro F1 0.5704
15 0.4243 val macro F1 0.6545
16 0.4295 val macro F1 0.593
17 0.5065 val macro F1 0.6489
18 0.4858 val macro F1 0.5721
19 0.4407 val macro F1 0.6182
20 0.3938 val macro F1 0.5749
21 0.4606 val macro F1 0.681
22 0.4378 val macro F1 0.5993
23 0.3978 val macro F1 0.5808
24 0.4241 val macro F1 0.6283
25 0.3411 val macro F1 0.636
26 0.3821 val macro F1 0.5994
27 0.4516 val macro F1 0.6183
28 0.3631 val macro F1 0.6225
29 0.4823 val macro F1 0.5815
best epoch 21
                               precision    recall  f1-score   support

                    Blackma